# Milestone 2 - DL GenAI

In [1]:
!pip install -q datasets transformers sentence-transformers torch

In [2]:
import numpy as np
import pandas as pd
import torch
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer, util
from transformers import BertTokenizer, AutoModel, pipeline
from datasets import load_dataset

## question 1

In [3]:
raw_dataset = load_dataset('csv', data_files='/content/train.csv')['train']

def build_combo(example):
    return {'text_blob': example['prompt'] + ' ' + example['A']}

raw_dataset = raw_dataset.map(build_combo)

record = raw_dataset[51]
print(record['text_blob'])
print()
print(len(record['text_blob']))

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Determine the correct option: What is the reason behind the designation of Class L dwarfs, and what is their color and composition? among the listed options. Class L dwarfs are hotter than M stars and are designated L because L is the remaining letter alphabetically closest to M. They are bright blue in color and are brightest in ultraviolet. Their atmosphere is hot enough to allow metal hydrides and alkali metals to be prominent in their spectra. Some of these objects have masses large enough to support hydrogen fusion and are therefore stars, but most are of substellar mass and are therefore brown dwarfs.

614


## question 2

In [4]:
wp_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
print(wp_tokenizer.vocab_size)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

30522


## question 3

In [5]:
sep_id_value = wp_tokenizer.sep_token_id
print("sep token id is", sep_id_value)

sep token id is 102


## question 4

In [6]:
table = pd.read_csv('/content/train.csv')

all_prompts = list(table['prompt'])
batch_enc = wp_tokenizer(all_prompts, padding='max_length', truncation=True, max_length=128, return_tensors='pt')

print(batch_enc['input_ids'].shape)

torch.Size([2000, 128])


## question 5

In [7]:
config = {'hidden': 768, 'heads': 12}
per_head = config['hidden'] / config['heads']
print(per_head)

64.0


## question 6

In [8]:
encoder = AutoModel.from_pretrained('bert-base-uncased')
encoder.eval()

p_zero = table.loc[0, 'prompt']
enc_zero = wp_tokenizer(p_zero, return_tensors='pt')

with torch.no_grad():
    output_zero = encoder(**enc_zero)

states = output_zero.last_hidden_state
print(states.shape)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([1, 31, 768])


## question 7

In [9]:
cls_embedding = states[0][0]
partial_sum = sum(cls_embedding[k].item() for k in range(5))
print(round(partial_sum, 4))

-1.2001


## question 8

In [10]:
encoder_attn = AutoModel.from_pretrained('bert-base-uncased', output_attentions=True)
encoder_attn.eval()

test_sentence = "Light-ion fusion is a technique."
enc_test = wp_tokenizer(test_sentence, return_tensors='pt')
piece_tokens = wp_tokenizer.convert_ids_to_tokens(enc_test['input_ids'][0])
print(piece_tokens)

word_index = piece_tokens.index('fusion')

with torch.no_grad():
    output_test = encoder_attn(**enc_test)

deepest_layer_attn = output_test.attentions[-1]
weight = deepest_layer_attn[0][0][0][word_index].item()
print()
print(round(weight, 4))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


['[CLS]', 'light', '-', 'ion', 'fusion', 'is', 'a', 'technique', '.', '[SEP]']

0.1025


## question 9

In [11]:
sbert = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

first_row = table.loc[0]
v_prompt = sbert.encode(first_row['prompt'])
v_optionB = sbert.encode(str(first_row['B']))

cos_val = util.cos_sim(v_prompt, v_optionB)
print(round(cos_val.item(), 4))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

0.7658


## question 10

In [12]:
letters = ['A', 'B', 'C', 'D', 'E']

def map_at_3(truth_list, guesses_list):
    acc = 0.0
    for gold, guesses in zip(truth_list, guesses_list):
        for rank, g in enumerate(guesses[:3]):
            if g == gold:
                acc += 1.0 / (rank + 1)
                break
    return acc / len(truth_list)

table['blob'] = [
    row['prompt'] + ''.join(' ' + str(row[c]) for c in letters)
    for _, row in table.iterrows()
]

vec = TfidfVectorizer(stop_words='english')
vec.fit(table['blob'])

tfidf_guesses = []
for _, row in table.iterrows():
    q_vec = vec.transform([row['prompt']])
    scored = []
    for c in letters:
        a_vec = vec.transform([str(row[c])])
        scored.append(cosine_similarity(q_vec, a_vec)[0][0])
    best3 = np.argsort(scored)[::-1][:3]
    tfidf_guesses.append([letters[j] for j in best3])

tfidf_result = map_at_3(table['answer'].tolist(), tfidf_guesses)
print("tfidf map3:", tfidf_result)

tfidf map3: 0.2552499999999985


In [13]:
prompt_embs = sbert.encode(list(table['prompt']), batch_size=64, show_progress_bar=True)

sbert_guesses = []
for i, (_, row) in enumerate(table.iterrows()):
    opts = [str(row[c]) for c in letters]
    opt_embs = sbert.encode(opts)
    scores = util.cos_sim(prompt_embs[i], opt_embs)[0].tolist()
    best3 = np.argsort(scores)[::-1][:3]
    sbert_guesses.append([letters[j] for j in best3])

sbert_result = map_at_3(table['answer'].tolist(), sbert_guesses)
print("minilm map3:", sbert_result)

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

minilm map3: 0.42308333333333487


In [14]:
wins = 0
for i, gold in enumerate(table['answer']):
    if gold not in tfidf_guesses[i] and gold in sbert_guesses[i]:
        wins += 1

print("count where minilm got it right but tfidf missed:", wins)

count where minilm got it right but tfidf missed: 613


## question 11

In [15]:
zsc = pipeline('zero-shot-classification')

second_row = table.loc[1]
label_set = [str(second_row['A']), str(second_row['B']), str(second_row['C'])]

single_result = zsc(second_row['prompt'], candidate_labels=label_set)
print(single_result)
print()
top1 = single_result['scores'][0]
print(round(top1, 4))

[transformers] No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

{'sequence': 'What is accelerator-based light-ion fusion?', 'labels': ['Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 100 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion

## question 12

In [16]:
multi_result = zsc(second_row['prompt'], candidate_labels=label_set, multi_label=True)
print(multi_result)
print()
sm_total = sum(single_result['scores'])
sg_total = sum(multi_result['scores'])

gap = abs(sm_total - sg_total)
print(round(gap, 4))

{'sequence': 'What is accelerator-based light-ion fusion?', 'labels': ['Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion 

## question 13

In [17]:
t2t = pipeline('text-generation', model='google/flan-t5-small')

zero_row = table.loc[0]
prompt_text = "Question: " + zero_row['prompt'] + ". Is the correct answer A: " + str(zero_row['A']) + " or B: " + str(zero_row['B']) + "? Answer with just the letter A or B."
print(prompt_text)
print()
gen = t2t(prompt_text, max_new_tokens=5)
print(gen[0]['generated_text'])

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

...

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DeepseekV32ForCausalLM', 'DeepseekV4ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCaus

Question: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.. Is the correct answer A: Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time. or B: Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.? Answer with just the letter A or B.

Question: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.. Is the correct answer A: Martin Heidegger believes that 